# 16. Document Augmentation

Enrich the index with generated questions that point back to the original chunk.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

Users phrase things differently from documents. **Document augmentation** adds extra, question-shaped entry points to the index: for each chunk the LLM writes questions it could answer, and those questions are indexed *alongside* the original chunk. The retriever can match either form and always returns the original text.

## How it works

1. Index every original chunk
2. For each chunk generate N questions
3. Index the questions too, tagged with the parent chunk id
4. At query time search the combined index
5. Deduplicate by parent chunk and return the original text

```text
 chunk c7 ---------------------------> index entry (type=chunk, parent=7)
   |--LLM--> "What is HNSW?" ---------> index entry (type=question, parent=7)
   |--LLM--> "How do ANN indexes work?" > index entry (type=question, parent=7)
 query --> top entries --> map to parents --> dedupe --> chunks
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [1]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

W1028 12:26:10.739000 14059 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


19 chunks indexed


In [2]:
def make_questions(chunk, n=3):
    data = llm_json(f"Write {n} questions a user might ask that this passage answers. Reply with a JSON list of strings.\n\nPassage: {chunk}")
    return [q for q in (data or []) if isinstance(q, str)]

aug_texts, aug_meta = [], []
for i, chunk in enumerate(chunks):
    aug_texts.append(chunk); aug_meta.append({"chunk": i, "kind": "chunk"})
    for q in make_questions(chunk):
        aug_texts.append(q); aug_meta.append({"chunk": i, "kind": "question"})

aug_store = build_store(aug_texts, aug_meta)
print(f"{len(chunks)} chunks + {len(aug_texts) - len(chunks)} generated questions")

19 chunks + 51 generated questions


Unlike HyPE, the original chunk vectors stay in the index; questions are *extra* entry points. Deduplicate by source chunk at query time.

In [3]:
def augmented_search(query, k=3):
    seen, out = set(), []
    for h in aug_store.search(query, k * 4):
        c = h["meta"]["chunk"]
        if c not in seen:
            seen.add(c)
            out.append({"idx": c, "text": chunks[c], "score": h["score"], "meta": h["meta"]})
    return out[:k]

query = "When should I use a smaller model in a RAG pipeline?"
hits = augmented_search(query)
for h in hits:
    print(h["meta"]["kind"], f"{h['score']:.3f}", h["text"][:100])
print(answer(query, [h["text"] for h in hits]))

question 0.661 # The Applied AI Handbook

## Introduction to RAG

Retrieval-Augmented Generation (RAG) combines a r
question 0.536 cuments and falls back to web search when they are irrelevant. Self-RAG trains the model to emit ref
question 0.501 best for facts that change or must be traceable to a source. Updating a RAG system is as simple as r


I don't know. The provided context does not mention when to use a smaller model in a RAG pipeline. It only discusses the general concept of RAG, its stages, fine-tuning versus RAG, and caching and latency.


## Which entry type wins?
Count how often the best match is the original chunk vs a generated question.

In [4]:
from collections import Counter
wins = Counter()
for q in load_eval_questions():
    wins[aug_store.search(q["question"], 1)[0]["meta"]["kind"]] += 1
print(wins)

Counter({'question': 14, 'chunk': 6})


## Evaluate against plain retrieval

In [5]:
compare_retrievers({"chunks only": store.search, "chunks + generated questions": augmented_search}, k=3)

,hit@1,hit@3,MRR
method,,,
chunks only,0.80,1.00,0.89
chunks + generated questions,0.75,0.95,0.84


## Show a few matches with the question that triggered them

In [6]:
for q in ["Why not just use a big context window?", "tell me about the ANN index types"]:
    print("Q:", q)
    for h in aug_store.search(q, 3):
        print(f"   {h['meta']['kind']:8s} {h['score']:.3f} {h['text'][:90]}")

Q: Why not just use a big context window?
   chunk    0.317 hunking Strategies

Documents must be split before indexing because embedding a whole book
   chunk    0.285 delay.

## Security and Privacy

Retrieved text is untrusted input. A malicious document c
   chunk    0.235 the embedding. A chunk size of around 500 to 1000 characters with 10 to 20 percent overlap
Q: tell me about the ANN index types
   question 0.478 What are approximate nearest neighbour indexes?
   chunk    0.345 res the query with every vector, which is accurate but slow at scale. Approximate nearest 
   question 0.303 What are the strengths of dense retrieval?


## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Closes the vocabulary gap between users and documents | Index grows 3-5x |
| Original chunk stays intact | LLM call per chunk at indexing |
| Entry points can be added incrementally | Poor questions create false matches |

## When to use it

Support/FAQ-style retrieval where users ask in their own words and the corpus is relatively stable.

## Experiments to try

- Generate keywords or summaries as additional entry points.
- Weight question matches lower than chunk matches.
- Add the document title to every generated question.

## Key takeaways

- Index multiple views of the same content.
- Always return the original chunk.
- Deduplicate by parent.

---
**Next:** [17. Fusion Retrieval (BM25 + Vectors)](../17_fusion_retrieval/fusion_retrieval.ipynb)